# Train Final GNN

Trains one STGATv2 on every annotated clip with no val split, using the hyperparameters validated by the 5-fold CV in `gnn.ipynb`. Output: `runs/gnn/final.pt`, the model for the live inference pipeline.

No early stopping (no val set to monitor). Epoch count comes from `cfg.gnn.epochs`.

## Imports

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [2]:
sys.path.insert(0, str(Path.cwd().parent))

In [3]:
from src.utils.config.config import Config
from src.classifier.common.constants import ACTION_NAMES, NUM_NODES, NUM_KEYPOINTS, OBJECT_NODE_IDX
from src.classifier.common.graph import build_adjacency
from src.classifier.common.augment import SkeletonAugment
from src.classifier.gnn.dataset import GNNHOIDataset, NODE_CHANNELS
from src.classifier.gnn.model import STGATv2

## Config

In [ ]:
SEED = 42

config_loader = Config()
cfg = config_loader.load_config()
hp = cfg.gnn

dataset_root = cfg.project_root / cfg.paths.dataset / "CafeV1"
clips_root   = dataset_root / "Clips"
cache_root   = dataset_root / "cache" / "keypoints"
folds_path   = dataset_root / "folds.json"

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"device: {cfg.device}")
print(f"checkpoint will land at: {hp.model_ckpt}")
print(f"hp: {hp}")

## Dataset

Every clip listed in `folds.json` is used. Same windowing and augmentations as the CV training in `gnn.ipynb`.

In [5]:
with open(folds_path) as f:
    folds_data = json.load(f)

all_clips = sorted(folds_data["assignments"].keys(),
                   key=lambda k: (int(k.split("/")[0]), int(k.split("/")[1])))

augmenter = SkeletonAugment(
    hflip_p=hp.hflip_p,
    jitter_std=hp.jitter_std,
    obj_dropout_p=hp.obj_dropout_p,
    num_keypoints=NUM_KEYPOINTS,
    object_node_idx=OBJECT_NODE_IDX,
)

full_set = GNNHOIDataset(
    clips_root, cache_root, all_clips, ACTION_NAMES,
    seq_len=hp.seq_len, stride=hp.stride, augment=augmenter,
)

print(f"clips: {len(all_clips)}")
print(f"windows: {len(full_set)}")
print(f"per class: {dict(zip(ACTION_NAMES, full_set.label_counts()))}")

clips: 126
windows: 3549
per class: {'idle': 1987, 'using_laptop': 71, 'using_phone': 987, 'reading': 504}


## Model

In [6]:
adj = build_adjacency(NUM_NODES).to(cfg.device)

model = STGATv2(
    in_channels=NODE_CHANNELS,
    num_classes=len(ACTION_NAMES),
    hidden=hp.hidden,
    heads=hp.heads,
    dropout=hp.dropout,
).to(cfg.device)

print(f"params: {sum(p.numel() for p in model.parameters()):,}")

params: 87,876


## Train

No val set so no early stopping. Loop runs for `hp.epochs` epochs. Class-weighted cross entropy because `using_laptop` is rare.

In [7]:
counts = full_set.label_counts()
total = sum(counts)
K = len(ACTION_NAMES)
class_weights = torch.tensor(
    [total / (K * max(1, c)) for c in counts],
    dtype=torch.float32, device=cfg.device,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=hp.lr, weight_decay=hp.weight_decay)

loader = DataLoader(full_set, batch_size=hp.batch_size, shuffle=True)

print(f"class weights: {class_weights.cpu().tolist()}")

class weights: [0.4465274214744568, 12.496479034423828, 0.8989361524581909, 1.7604166269302368]


In [8]:
for epoch in range(1, hp.epochs + 1):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for x, y in loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)

        logits = model(x, adj)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n += x.size(0)

    print(f"epoch {epoch:3d} | loss {total_loss/total_n:.4f} | acc {total_correct/total_n:.3f}")

epoch   1 | loss 0.8401 | acc 0.766
epoch   2 | loss 0.3768 | acc 0.929
epoch   3 | loss 0.3714 | acc 0.936
epoch   4 | loss 0.3652 | acc 0.935
epoch   5 | loss 0.3190 | acc 0.942
epoch   6 | loss 0.3342 | acc 0.937
epoch   7 | loss 0.3473 | acc 0.937
epoch   8 | loss 0.3663 | acc 0.938
epoch   9 | loss 0.3397 | acc 0.935
epoch  10 | loss 0.3853 | acc 0.935
epoch  11 | loss 0.3148 | acc 0.939
epoch  12 | loss 0.3262 | acc 0.941
epoch  13 | loss 0.3070 | acc 0.937
epoch  14 | loss 0.2986 | acc 0.941
epoch  15 | loss 0.3661 | acc 0.926
epoch  16 | loss 0.3099 | acc 0.935
epoch  17 | loss 0.3299 | acc 0.937
epoch  18 | loss 0.3147 | acc 0.937
epoch  19 | loss 0.2990 | acc 0.933
epoch  20 | loss 0.2919 | acc 0.935
epoch  21 | loss 0.3107 | acc 0.944
epoch  22 | loss 0.2769 | acc 0.943
epoch  23 | loss 0.3004 | acc 0.933
epoch  24 | loss 0.2701 | acc 0.945
epoch  25 | loss 0.2877 | acc 0.941
epoch  26 | loss 0.2558 | acc 0.946
epoch  27 | loss 0.2943 | acc 0.929
epoch  28 | loss 0.3012 | ac

## Save

This is the checkpoint the live inference pipeline will load.

In [10]:
save_path = cfg.gnn.model_ckpt
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f"saved {save_path}")

AttributeError: 'GNNConfig' object has no attribute 'model_ckpt'